In [ ]:
# Best Neural Network: TF-IDF + MLP (Dense Layers)

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import accuracy_score, classification_report

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1️⃣ Load data
data = fetch_20newsgroups(subset='all', shuffle=True, random_state=42)
X, y = data.data, data.target
target_names = data.target_names

# 2️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3️⃣ Convert text → high-dimensional TF-IDF features
tfidf = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    ngram_range=(1,2),
    max_features=50000     # control dimensionality (50k-100k)
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# 4️⃣ Convert sparse matrix → dense arrays (needed for Keras)
X_train_dense = X_train_tfidf.toarray().astype(np.float32)
X_test_dense  = X_test_tfidf.toarray().astype(np.float32)

# One-hot encode labels
encoder = LabelBinarizer()
y_train_oh = encoder.fit_transform(y_train)
y_test_oh  = encoder.transform(y_test)

# 5️⃣ Define Neural Network
model = keras.Sequential([
    layers.Input(shape=(X_train_dense.shape[1],)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(target_names), activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 6️⃣ Train
history = model.fit(
    X_train_dense, y_train_oh,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    verbose=2
)

# 7️⃣ Evaluate
test_loss, test_acc = model.evaluate(X_test_dense, y_test_oh, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

# Predict and print detailed metrics
y_pred = np.argmax(model.predict(X_test_dense), axis=1)
print(classification_report(y_test, y_pred, target_names=target_names))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 512)                 │      25,600,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 20)                  │           5,140 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 25,736,980 (98.18 MB)

 Trainable params: 25,736,980 (98.18 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
106/106 - 30s - 287ms/step - accuracy: 0.6834 - loss: 1.4956 - val_accuracy: 0.9125 - val_loss: 0.3775
Epoch 2/10
106/106 - 28s - 266ms/step - accuracy: 0.9631 - loss: 0.1571 - val_accuracy: 0.9211 - val_loss: 0.2765
Epoch 3/10
